# **Information bottleneck attribution: a map in bits with our own code**

Practice for the module ["Axiomatic attributions"](https://ai-interpretability.school), lesson "Wrap-up of gradient-based methods", step "What the methods of the block share".

Information bottleneck attribution, or IBA ([Schulz et al., 2020](https://arxiv.org/abs/2001.00396)), defines importance not through a derivative and not through a difference with a reference point, but through capacity: noise is inserted into one layer of the network, and for every position one finds how much information about the input has to pass through it for the class to stay the same. Here the method is written in our own torch code, about thirty lines, and compared with Integrated Gradients on the same image and the same model as in the notebook [HW9_integrated_gradients](https://colab.research.google.com/github/SadSabrina/open-xai-materials/blob/main/homeworks/HW9_integrated_gradients_en.ipynb):

1. ResNet-50 and the photo of a cat in a kitchen;
2. means and spreads of the layer's activations over other ImageNet photos: without them the noise has nothing to be drawn from;
3. the bottleneck itself and the map in bits;
4. comparison with the IG map: correlation and the share of mass inside the cat's box;
5. sensitivity to $\beta$, to the choice of layer, to the class and to the noise seed.

Runs in about three minutes on a CPU.

In [ ]:
!pip install -q captum==0.9.0

In [ ]:
import io
import math
import tarfile
import time
import urllib.request
import warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # tqdm without ipywidgets; Colab has ipywidgets

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet50
from captum.attr import IntegratedGradients

torch.manual_seed(0)
DATA = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main'


def fetch(path):
    return urllib.request.urlopen(f'{DATA}/' + path, timeout=30).read()


model = resnet50(weights='IMAGENET1K_V1').eval()
for p in model.parameters():
    p.requires_grad_(False)

# the same preprocessing as in HW9_integrated_gradients
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
photo = Image.open(io.BytesIO(fetch('data/cat.jpg'))).convert('RGB')
x = tf(photo).unsqueeze(0)
names = [s.strip() for s in fetch('data/imagenet_classes.txt').decode().split('\n')]

with torch.no_grad():
    probs = model(x).softmax(-1)[0]
for p, i in zip(*probs.topk(3)):
    print(f'{int(i):4d}  {names[i]:15}  {float(p):.3f}')

The model sees the cat with confidence: class `tiger cat` (282). This is the class we will explain.

## 1. Means and spreads of activations

The noise in the bottleneck is not arbitrary: it is drawn from the same distribution as the layer's activations on ordinary photos, with its own mean $\mu$ and spread $\sigma$ for every channel and every position. Replacing an activation with such noise then erases the information about **our** photo without pushing the layer out of its usual regime. The authors estimate $\mu$ and $\sigma$ on the ImageNet training set; here we use [Imagenette](https://github.com/fastai/imagenette), an open subset of ten ImageNet classes by fast.ai. The archive is read as a stream: its validation part comes first, every twentieth photo is taken from it, 197 in total, and reading stops before the training part.

The moments are computed for three ResNet-50 layers at once: `layer1`, `layer2` and `layer3`. The main layer is `layer2`, the output of the conv3 block of size $512\times28\times28$: this is where the authors show their maps for ResNet-50.

In [ ]:
LAYERS = {'layer1': model.layer1, 'layer2': model.layer2, 'layer3': model.layer3}
URL = 'https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz'

start = time.time()
images = []
with urllib.request.urlopen(URL, timeout=60) as stream:
    tar = tarfile.open(fileobj=stream, mode='r|gz')
    k = 0
    for member in tar:
        if '/train/' in member.name:
            break                                   # the validation part is over, stop reading
        if member.name.endswith('.JPEG'):
            if k % 20 == 0:                         # every twentieth photo
                images.append(tf(Image.open(tar.extractfile(member)).convert('RGB')))
            k += 1
print(f'photos in the validation part: {k}, taken: {len(images)}, {time.time() - start:.0f} s')

# sums of activations and of their squares over photos, in double so that E[a^2] - E[a]^2 keeps its precision
sums = {name: [0.0, 0.0] for name in LAYERS}
feats = {}
hooks = [m.register_forward_hook(lambda mod, inp, out, name=name: feats.__setitem__(name, out.double()))
         for name, m in LAYERS.items()]
with torch.no_grad():
    for i in range(0, len(images), 32):
        model(torch.stack(images[i:i + 32]))
        for name, a in feats.items():
            sums[name] = [sums[name][0] + a.sum(0), sums[name][1] + (a ** 2).sum(0)]
for h in hooks:
    h.remove()

N = len(images)
MU = {name: (s / N).float() for name, (s, s2) in sums.items()}
# the spread is bounded below by 0.01, as in the authors' code: otherwise activations that are always zero after ReLU divide by zero
STD = {name: (s2 / N - (s / N) ** 2).clamp(min=0).sqrt().float().clamp(min=0.01) for name, (s, s2) in sums.items()}
for name in LAYERS:
    print(f'{name}: activation shape {tuple(MU[name].shape)}')

Let $R$ be the activations of the chosen layer on our photo. The bottleneck replaces them with a mixture with noise:

$$Z=\lambda\odot R+(1-\lambda)\odot\varepsilon,\qquad \varepsilon\sim\mathcal N(\mu,\sigma^2),\qquad \lambda=\sigma_{\text{sig}}(\alpha)\in(0,1),$$

where $\lambda$ is a separate share for every channel and every position. At $\lambda=1$ the activation passes in full, at $\lambda=0$ nothing of the photo is left, only noise. How much information about $R$ is left in $Z$ is estimated by the Kullback–Leibler divergence between the distribution of $Z$ given $R$ and the noise distribution. In normalized coordinates $r=(R-\mu)/\sigma$ this Gaussian divergence takes one line:

$$I_{ij}=\tfrac12\left(\lambda_{ij}^2r_{ij}^2+(1-\lambda_{ij})^2-1\right)-\log(1-\lambda_{ij}).$$

The parameters $\alpha$ are fitted to one photo by optimizing

$$\mathcal L=\mathrm{CE}\big(f(Z),\,c\big)+\beta\cdot\overline{I},$$

where the first term requires class $c$ to stay the same and the second, the mean of $I$ over all elements of the layer, penalizes every bit let through. The settings follow the authors: $\alpha$ starts at 5 ($\lambda\approx0.993$, almost everything passes), 10 Adam steps with step size 1, 10 copies of the photo with their own noise at every step, $\beta=10$, $\lambda$ smoothed over positions with a Gaussian kernel with $\sigma=1$. The map is the sum of $I$ over channels, converted to bits by dividing by $\ln 2$.

In [ ]:
def gaussian_kernel(sigma=1.0):
    # a 7x7 kernel, as in the authors' code for sigma = 1
    r = torch.arange(-3, 4, dtype=torch.float32)
    g = torch.exp(-r ** 2 / (2 * sigma ** 2))
    g = g / g.sum()
    return g[:, None] * g[None, :]


KERNEL = gaussian_kernel()


def smooth(lam):
    # smoothing the share lambda over positions, separately in every channel
    c = lam.shape[1]
    return F.conv2d(F.pad(lam, (3, 3, 3, 3), mode='reflect'), KERNEL.expand(c, 1, 7, 7), groups=c)


def iba(x, target, layer='layer2', beta=10.0, steps=10, batch=10, lr=1.0, seed=0):
    # a map in bits: how much information about the input each position of the layer lets through for class target to stay
    mu, std = MU[layer], STD[layer]
    gen = torch.Generator().manual_seed(seed)
    alpha = torch.full((1,) + tuple(mu.shape), 5.0, requires_grad=True)
    opt = torch.optim.Adam([alpha], lr=lr)
    state = {}

    def bottleneck(module, inputs, R):
        lam = smooth(torch.sigmoid(alpha))
        eps = mu + std * torch.randn(R.shape, generator=gen)
        r = (R - mu) / std
        state['I'] = 0.5 * (lam ** 2 * r ** 2 + (1 - lam) ** 2 - 1) - torch.log(1 - lam)
        return lam * R + (1 - lam) * eps

    hook = LAYERS[layer].register_forward_hook(bottleneck)
    xs = x.expand(batch, -1, -1, -1)
    labels = torch.full((batch,), target)
    for _ in range(steps):
        opt.zero_grad()
        loss = F.cross_entropy(model(xs), labels) + beta * state['I'].mean()
        loss.backward()
        opt.step()
    with torch.no_grad():
        model(x)                                   # the last pass: I at the final lambda
    hook.remove()
    return state['I'][0].sum(0) / math.log(2)


TARGET = 282
start = time.time()
bits = iba(x, TARGET)
print(f'time: {time.time() - start:.1f} s, map {tuple(bits.shape)}')
print(f'let through in total: {float(bits.sum()):.0f} bits, maximum at one position: {float(bits.max()):.0f} bits')

## 3. Comparison with Integrated Gradients

The IG map for the same class: `captum.attr.IntegratedGradients`, zero baseline, 50 steps, as in HW9. To compare on one grid, the absolute IG attributions are summed over channels and averaged over $8\times8$ pixel squares, which gives the same $28\times28$ grid as that of `layer2`.

Two numbers of comparison: the Pearson correlation of the two maps and the share of a map's mass that falls inside the cat's box. The box is marked by eye on the $224\times224$ photo and covers about a tenth of the frame: a map spread evenly over the frame would put the same share into it.

In [ ]:
ig = IntegratedGradients(model).attribute(x, baselines=x * 0, target=TARGET, n_steps=50, internal_batch_size=10)
ig_map = F.avg_pool2d(ig.abs().sum(1, keepdim=True), 8)[0, 0]      # 224x224 -> 28x28

CAT_BOX = (90, 55, 132, 180)          # x0, y0, x1, y1 on the 224x224 photo, marked by eye


def corr(a, b):
    a, b = a.flatten() - a.mean(), b.flatten() - b.mean()
    return float((a * b).sum() / (a.norm() * b.norm()))


def mass_in(m, box=CAT_BOX):
    # share of a map's mass inside the box; the map is stretched to 224x224
    big = F.interpolate(m[None, None], (224, 224), mode='bilinear', align_corners=False)[0, 0]
    x0, y0, x1, y1 = box
    return float(big[y0:y1, x0:x1].sum() / big.sum())


area = (CAT_BOX[2] - CAT_BOX[0]) * (CAT_BOX[3] - CAT_BOX[1]) / 224 ** 2
print(f'the cat box covers {area * 100:.1f}% of the frame')
print(f'share of mass inside the cat box: IBA {mass_in(bits) * 100:.1f}%, IG {mass_in(ig_map) * 100:.1f}%')
print(f'correlation of the IBA and IG maps: {corr(bits, ig_map):.3f}')

IBA puts several times more mass into the cat's box than IG does, and several times more than an even map would. The correlation of the two maps is nevertheless low: IG gives a noticeable share of attribution to the kitchen background, while IBA drowns almost all of the background in noise, and the class does not change.

The reason is the question each method asks. IG answers how much each pixel added to the logit along the path from a black image; the background also lies on this path, and its contribution is not zero. IBA answers what the class cannot be kept without: the model does not need the background once the information about the cat got through.

## 4. The price of information $\beta$

$\beta$ is the only substantive parameter of the method. The larger it is, the more each bit costs, and the less information the bottleneck lets through.

In [ ]:
print(f'{"beta":>6}{"total bits":>12}{"in cat box":>14}{"corr. w/ IG":>13}')
for beta in (1, 10, 100):
    m = iba(x, TARGET, beta=beta)
    print(f'{beta:>6}{float(m.sum()):>12.0f}{mass_in(m) * 100:>13.1f}%{corr(m, ig_map):>13.3f}')

As $\beta$ grows, the total number of bits let through goes down, and their share inside the cat's box goes up: the bottleneck keeps only what is necessary. A map in bits is therefore not an absolute quantity: the number of bits and the shape of the map are reported together with $\beta$.

## 5. Choice of layer

The bottleneck can be inserted into any layer. An early layer sees fine detail, and its grid is dense: $56\times56$ for `layer1`. The late `layer3` has a $14\times14$ grid, but each of its positions already knows about a large piece of the photo.

In [ ]:
layer_maps = {}
print(f'{"layer":>8}{"grid":>10}{"total bits":>12}{"in cat box":>14}')
for name in LAYERS:
    layer_maps[name] = iba(x, TARGET, layer=name)
    m = layer_maps[name]
    print(f'{name:>8}{m.shape[0]:>7}x{m.shape[1]:<2}{float(m.sum()):>12.0f}{mass_in(m) * 100:>13.1f}%')

The layer changes both the grid of the map and how much mass lands on the cat. As with LRP and its rules, the layer is reported together with the map.

## 6. Which class was asked about

Unlike a raw attention map, IBA depends on the class: the penalty for bits is the same, but a different answer has to be kept.

**Task.** Build the IBA map at `layer2` with $\beta=10$ for class `tabby` (281, also a cat) and for class `dining table` (532, the table the cat sits on). For each, print the total number of bits let through, the share of mass inside the cat's box and the correlation with the map for `tiger cat`. Which of the two classes required letting more information through, and where did the mass of the map go?

In [ ]:
# Your code here

## 7. The map depends on the noise seed

The bottleneck is optimized over random noise, so two runs with different seeds give different maps. How different, we check on three seeds.

In [ ]:
print(f'{"seed":>4}{"total bits":>12}{"in cat box":>14}{"corr. w/ seed 0":>18}')
for seed in (0, 1, 2):
    m = iba(x, TARGET, seed=seed)
    print(f'{seed:>4}{float(m.sum()):>12.0f}{mass_in(m) * 100:>13.1f}%{corr(m, bits):>18.4f}')

The shape of the map hardly depends on the seed: the correlation with the map of seed 0 is above 0.99. The total number of bits let through, however, differs between seeds by more than a tenth, and the share of mass inside the cat's box by a few percentage points. This is the cost of optimizing for every object: IG with the same number of steps gives the same result on every run, while for IBA the noise seed belongs to the description of the map just like $\beta$ and the layer. On a GPU the noise generator draws other random numbers for the same optimization, and the numbers of this notebook will match there only approximately.

## 8. Look with your own eyes

In [ ]:
panels = [('photo', None), ('IBA, layer2, beta = 10', bits), ('IG, zero baseline', ig_map),
          ('IBA, layer3', layer_maps['layer3']), ('IBA, class dining table', iba(x, 532))]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (title, m) in zip(axes, panels):
    ax.imshow(photo.resize((224, 224)))
    if m is not None:
        big = F.interpolate(m[None, None], (224, 224), mode='bilinear', align_corners=False)[0, 0]
        ax.imshow(big.numpy(), cmap='inferno', alpha=0.7)
    x0, y0, x1, y1 = CAT_BOX
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color='cyan', lw=1.5))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## What to take away

- **IBA asks what the class cannot be kept without.** A position is important if it cannot be replaced with noise without losing the answer; background the model can do without gets almost zero bits, even when IG gives it a noticeable attribution.
- **The noise is drawn from the distribution of the layer's activations.** Without means and spreads over other photos the method is undefined; here they came from Imagenette, in the paper from the ImageNet training set.
- **A map is reported together with $\beta$, the layer and the noise seed.** $\beta$ changes both the number of bits and the share on the object, the layer changes the grid and the shape of the map, the seed changes the total of bits and the share of mass inside the box.
- **The sum of the map is not compared with the difference of predictions.** A map in bits does not obey completeness: this is where IBA differs from IG, DeepLIFT and LRP.
- **Optimization for every object.** Ten optimization steps, each a forward and a backward pass over ten copies of the photo: a hundred forward-and-backward passes of the photo per map, not counting the estimate of the moments. IG with 50 steps needs fifty.